# Combined MR Summary Analysis

This notebook loads all MR summary CSV files from both b_scans and c_scans,
combines them into a single dataframe, and prepares the data for comprehensive analysis.

## Analysis Components:
1. Load all MR_summary CSV files from b_scans and c_scans
2. Combine into a single dataframe with temperature and scan_type columns
3. Explore the combined dataset
4. Prepare for further analysis (MR vs T, I_min/I_max vs T, etc.)

## 1. Setup & Imports

In [ ]:
# Notebook setup
from scripts.utils import setup_notebook, COLOR_B_AXIS, COLOR_C_AXIS, OKABE_ITO_CYCLE
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()

# Uncertainty helpers used by the plots below: the error on the position of the MR
# maximum (a derived quantity, so the CSVs carry no column for it), the propagation
# of the measured current error through the I(V) transforms, and the empirical
# point-to-point scatter that floors both (the CSV errors are standard errors of
# the mean over correlated field points and are too small on their own).
from scripts.mr_peak_uncertainty import peak_voltage_with_error
from scripts.mr_trend_scatter import SMOOTH_T_RANGE_K, with_trend_floor
from scripts.iv_error_propagation import (
    d2idv2_with_error,
    didv_with_error,
    fill_sigma_band,
    fn_ordinate_error,
    normalized_didv_with_error,
)

# Additional imports
import glob
import re


## 2. Load All MR Summary CSV Files

In [ ]:
# Define paths to MR_summary directories
b_scans_dir = PROJECT_ROOT / r"output/IV_H_scans/MR_summary/b_scans"
c_scans_dir = PROJECT_ROOT / r"output/IV_H_scans/MR_summary/c_scans"

# Find all CSV files
b_scan_files = list(b_scans_dir.glob("TMR_ratio_vs_V_*.csv"))
c_scan_files = list(c_scans_dir.glob("TMR_ratio_vs_V_*.csv"))

print(f"Found {len(b_scan_files)} b_scan CSV files")
print(f"Found {len(c_scan_files)} c_scan CSV files")
print(f"Total: {len(b_scan_files) + len(c_scan_files)} files")

In [ ]:
def load_mr_csv_with_metadata(csv_path, scan_type):
    """
    Load a single MR summary CSV file and add temperature and scan_type columns.
    
    Parameters:
    -----------
    csv_path : Path
        Path to the CSV file
    scan_type : str
        Either 'b_scan' or 'c_scan'
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with added 'temperature' and 'scan_type' columns
    """
    # Extract temperature from filename (e.g., TMR_ratio_vs_V_10K.csv -> 10)
    filename = csv_path.name
    temp_match = re.search(r'(\d+)K', filename)
    
    if not temp_match:
        print(f"Warning: Could not extract temperature from {filename}")
        return None
    
    temperature = int(temp_match.group(1))
    
    # Load CSV
    df = pd.read_csv(csv_path)
    
    # Add metadata columns
    df['temperature'] = temperature
    df['scan_type'] = scan_type
    
    return df

# Load all b_scan files
b_scan_dataframes = []
for csv_file in b_scan_files:
    df = load_mr_csv_with_metadata(csv_file, 'b_scan')
    if df is not None:
        b_scan_dataframes.append(df)

# Load all c_scan files
c_scan_dataframes = []
for csv_file in c_scan_files:
    df = load_mr_csv_with_metadata(csv_file, 'c_scan')
    if df is not None:
        c_scan_dataframes.append(df)

print(f"\nSuccessfully loaded:")
print(f"  b_scans: {len(b_scan_dataframes)} dataframes")
print(f"  c_scans: {len(c_scan_dataframes)} dataframes")

## 3. Combine All Data into Single DataFrame

In [ ]:
# Combine all dataframes
all_dataframes = b_scan_dataframes + c_scan_dataframes
df_combined = pd.concat(all_dataframes, ignore_index=True)

print(f"\n{'='*70}")
print(f"Combined DataFrame Created!")
print(f"{'='*70}")
print(f"Total rows: {len(df_combined):,}")
print(f"Total columns: {len(df_combined.columns)}")
print(f"\nColumns: {list(df_combined.columns)}")
print(f"\nTemperature range: {df_combined['temperature'].min()}K to {df_combined['temperature'].max()}K")
print(f"Unique temperatures: {sorted(df_combined['temperature'].unique())}")
print(f"\nScan types: {df_combined['scan_type'].unique()}")
print(f"\nData shape: {df_combined.shape}")

In [ ]:
# Display first few rows
print("\nFirst 10 rows of combined data:")
df_combined.head(10)

In [ ]:
# Display summary statistics
print("\nSummary Statistics:")
df_combined.describe()

## 4. Data Quality Check

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df_combined.isnull().sum())
print(f"\nTotal missing values: {df_combined.isnull().sum().sum()}")
print(f"Percentage of missing data: {100 * df_combined.isnull().sum().sum() / (df_combined.shape[0] * df_combined.shape[1]):.2f}%")

In [ ]:
# Count data points per temperature and scan type
print("\nData points per temperature and scan type:")
pivot_counts = df_combined.groupby(['temperature', 'scan_type']).size().unstack(fill_value=0)
print(pivot_counts)

## 5. Save Combined DataFrame

In [ ]:
# Save combined dataframe for future use
output_dir = PROJECT_ROOT / r"output/IV_H_scans/MR_summary"
output_dir.mkdir(parents=True, exist_ok=True)

# Save as pickle for fast loading with data types preserved
pkl_path = output_dir / "MR_summary_combined_all_temps.pkl"
df_combined.to_pickle(pkl_path)
print(f"Saved combined dataframe (pickle): {pkl_path}")

# Save as CSV for portability
csv_path = output_dir / "MR_summary_combined_all_temps.csv"
df_combined.to_csv(csv_path, index=False)
print(f"Saved combined dataframe (CSV): {csv_path}")

## 6. Quick Visualization: MR vs Voltage for All Temperatures

### Where the error bars come from

Every data plot from here on carries an uncertainty. Four sources feed them:

- **Measured quantities.** `TMR_Error`, `I_par_error (A)` and `I_apar_error (A)`
  come straight from the summary CSVs. `TMR_Error` is the error on the *ratio*,
  so it is multiplied by 100 wherever the axis is in percent.
- **Transforms of the current** (Fowler-Nordheim ordinate, $dI/dV$, $d^2I/dV^2$,
  and the normalised $(dI/dV)/(I/V)$). `np.gradient` is linear, so its Jacobian is
  recovered exactly by differentiating the identity matrix on the same bias grid,
  and the current errors propagate through it. See
  `scripts/iv_error_propagation.py`. The propagation was checked against a
  Monte-Carlo resampling of the currents and agrees to better than 1 %.
- **The position of the MR maximum**, $V_\mathrm{bias}^\mathrm{max(MR)}$, which has
  no error column at all, because it is the *location* of an extremum rather than
  a measured value.
- **The empirical point-to-point scatter**, which floors all of the above. This is
  described in its own subsection below.

### Why the CSV errors are a floor, not the answer

`I_par_error (A)` is the standard error of the mean of the current over the field
points with $|H| > 0.5$ T, and `I_apar_error (A)` the same over $|H| < 0.1$ T. Those
field points are **not independent repeats**. Within one magnetic state the current
still drifts smoothly with field: the lag-1 autocorrelation of the residual in
acquisition order is 0.9 or higher for the $b$-axis plateau and 0.95 or higher for
the $c$-axis one. Dividing the spread by $\sqrt{N}$ therefore assumes far more
independent information than the measurement contains.

The consequence is specific rather than global. **Inside one MR-vs-bias curve** the
quoted error is about right: the point-to-point scatter along the bias axis is
comparable to it (median ratio near 1, printed below each figure). **From one
temperature to the next** it is far too small, because a temperature step means a
fresh cooldown, remagnetisation and re-measurement, and none of that reproducibility
enters an average over field points inside a single run. Two independent checks give
the size of the shortfall:

1. **Scatter about a smooth trend.** $\mathrm{max(MR)}(T)$ and
   $V_\mathrm{bias}^\mathrm{max(MR)}(T)$ vary smoothly with temperature over 20-90 K,
   so their point-to-point scatter there is noise. It is **3-7 times** the propagated
   error.
2. **$\sqrt{\chi^2_\mathrm{red}}$ of a smooth fit in $T$** weighted by the propagated
   errors: 3.5 ($b$) and 4.2 ($c$) for $\mathrm{max(MR)}$, 2.9 and 2.8 for
   $V_\mathrm{bias}^\mathrm{max(MR)}$. Same conclusion by the standard route.

Two further candidate estimators were tested and **rejected**, because in this
dataset they measure real physics rather than noise:

- **$b$-axis against $c$-axis.** The difference is systematic, not random: it closes
  monotonically from 283 % at 3 K to below 10 % above 80 K, with $b$ above $c$
  throughout. It is also not a like-for-like comparison. The $b$-axis parallel state
  is a genuine plateau (the current varies by 0.3-0.9 % over $|H| > 0.5$ T), whereas
  the $c$-axis sweep runs to 3.4 T and the current there still rises by 15-30 % across
  the same window, because the moments keep canting towards $H_\mathrm{sat}^\mathrm{c}$.
  Averaging $I_\mathrm{par}$ over $|H| > 0.5$ T therefore understates the $c$-axis MR.
- **Positive against negative bias branch.** This is the real interface asymmetry the
  manuscript attributes to the inequivalent top and bottom FLG/CrSBr contacts: mean
  $+67$ % in MR and $-49$ mV in peak bias over 20-90 K, both varying smoothly with $T$.

### Estimator for the empirical scatter

`scripts/mr_trend_scatter.py` measures it directly from each plotted series. The
second difference $y_{i-1} - 2y_i + y_{i+1}$ cancels any locally linear trend, and
for independent points of common $\sigma$ its variance is $6\sigma^2$, so
$\sigma = \mathrm{RMS}(\Delta^2 y)/\sqrt{6}$. Triples whose two abscissa spacings
differ are skipped, so the gap left by the current-noise cut around zero bias is not
read as noise. For the temperature panels the estimate is taken over **20-90 K**,
where the trend is smooth enough for the three-point model to hold, and applied to
the whole series; above about 100 K the MR collapse towards the zero-bias cusp is
steep enough that genuine curvature would inflate the estimate.

The measured values are **5.9 %** ($b$) and **4.6 %** ($c$) for $\mathrm{max(MR)}$,
and **17 mV** ($b$) and **16 mV** ($c$) for $V_\mathrm{bias}^\mathrm{max(MR)}$ on the
positive branch (27 mV and 27 mV on the negative branch). At print size, on a 6x5 in
panel at 300 dpi, that makes the bars comparable to the marker rather than
conspicuous: this is as large as the data honestly supports, and it is the reason the
points look reproducible.

Every error bar plotted below is then the **larger** of the propagated error and
this scatter. The maximum is conservative in both directions: it never shrinks a
propagated error that is legitimately large (the plateau fallback on the peak bias
above 100 K), and never lets one sit below the scatter the data actually shows.

### Estimator for the uncertainty on $V_\mathrm{bias}^\mathrm{max(MR)}$

The estimator used is a **local inverse-variance-weighted parabola fit**, implemented
in `scripts/mr_peak_uncertainty.py`:

1. Take the discrete argmax of `TMR_Ratio` on the bias branch of interest.
2. Fit $y = a(V - V_0)^2 + b(V - V_0) + c$ to every point within **$\pm 0.10$ V** of
   it, weighted by $1/\sigma^2$ from `TMR_Error`. That window holds about 19 points
   at the 10 mV bias step. Narrower windows are noise-dominated and the fitted
   curvature changes sign; wider ones leave the parabolic region and the vertex
   drifts systematically.
3. Report the vertex $V_0 - b/2a$ and propagate the fit covariance matrix to it.
   The resulting error is scaled by $\sqrt{\chi^2_\mathrm{red}}$ whenever
   $\chi^2_\mathrm{red} > 1$, so a peak shape that a parabola does not fully describe
   inflates the error bar instead of being ignored.
4. Floor the result at half the bias step: no estimator can localise a peak better
   than the sampling of the sweep.

The fit is accepted only if the curvature is negative, the vertex falls inside the
bias range it was fitted to, and the error is smaller than the fit window itself.
Where it is rejected, the maximum is not the interior extremum of a smooth peak, and
the documented **fallback** is used instead: the half-width of the bias interval over
which `TMR_Ratio` stays within one `TMR_Error` of its maximum. Every point records
which of the two produced it, and the cells below print that breakdown.

For this dataset the parabola resolves the peak over **20-90 K**, giving
$\sigma_V \approx 5$-$8$ mV. It is rejected below about 10 K, where the current-noise
cut removes most of the bias sweep, and above about 100 K, where the MR maximum has
collapsed onto the zero-bias cusp so that there is no interior peak left to fit.
That per-temperature $\sigma_V$ is then superseded by the 17 mV ($b$) and 16 mV ($c$)
point-to-point scatter of the series, for the reason given above.

### Caption-ready statement

> Error bars are the empirical point-to-point reproducibility of each series, taken
> as the root-mean-square scatter of the data about a locally linear trend in
> temperature over 20-90 K, and are never smaller than the uncertainty propagated
> from the measured currents.


In [ ]:
# ============================================================
# FILTER CONFIGURATION - Adjust these values as needed
# ============================================================

# Minimum I_apar threshold (A) - removes noisy low-current data
i_apar_threshold = 2e-9  # Remove data where |I_apar| < 1 nA

# Voltage cutoffs per temperature (K: max_voltage in V)
# Only temperatures listed here will have voltage limits applied
# Leave empty {} to show full voltage range for all temperatures


# Plot style configuration
plot_with_errorbars = True  # Set to False to plot without error bars (faster)
errorbar_alpha = 0.6  # Transparency of error bars (0-1)

# ============================================================


def apply_filters(data, temp, i_threshold, v_cutoffs):
    """
    Apply current and voltage filters to TMR data.
    
    Parameters:
    -----------
    data : pd.DataFrame
        DataFrame subset for a specific temperature/scan_type
    temp : int or float
        Temperature value
    i_threshold : float
        Minimum |I_apar| threshold in Amperes
    v_cutoffs : dict
        Dictionary of {temperature: max_voltage} for voltage cutoffs
    
    Returns:
    --------
    pd.DataFrame
        Filtered DataFrame
    """
    # Filter 1: Remove data where |I_apar| < threshold
    filtered = data[np.abs(data['I_apar (A)']) >= i_threshold].copy()
    
    # Filter 2: Apply voltage cutoff if specified for this temperature
    if temp in v_cutoffs:
        max_v = v_cutoffs[temp]
        filtered = filtered[np.abs(filtered['Voltage (V)']) <= max_v]
    
    return filtered
v_cutoffs = {}

temperatures = [20,30,40,50,60,70]

# ── Figure 1: b_scans ────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(6, 5))

for i, temp in enumerate(temperatures):
    
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'b_scan')]
    data = apply_filters(data, temp, i_apar_threshold, v_cutoffs={})
    if len(data) > 0:
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
        if plot_with_errorbars:
            ax1.errorbar(data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100,
                         yerr=data['TMR_Error'],
                         fmt='o', color=color, linewidth=1.5,
                         ecolor=color, elinewidth=0.5, capsize=0,
                         errorevery=5, alpha=errorbar_alpha,
                         label=f'{temp} K')
        else:
            ax1.plot(data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100, '-',
                     color=color, linewidth=1.5, alpha=0.7, label=f'{temp} K')

ax1.set_xlabel('$V_{\mathrm{bias}}$ (V)')
ax1.set_ylabel('MR ratio (%)')
#ax1.grid(False)
ax1.set_ylim(0, 400)
ax1.legend(loc='upper left', borderaxespad=0, frameon=True)

fig1.tight_layout()
plt.show()

# ── Figure 2: c_scans ────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(6, 5))

for i, temp in enumerate(temperatures):
    
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'c_scan')]
    data = apply_filters(data, temp, i_apar_threshold, v_cutoffs={})
    if len(data) > 0:
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
        if plot_with_errorbars:
            ax2.errorbar( data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100,
                         yerr=data['TMR_Error'],
                         fmt='o', color=color, linewidth=1.5,
                         ecolor=color, elinewidth=0.5, capsize=0,
                         errorevery=5, alpha=errorbar_alpha,
                         label=f'{temp} K')
        else:
            ax2.plot(data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100, '-',
                     color=color, linewidth=1.5, alpha=0.7, label=f'{temp} K')
ax2.set_xlabel('$V_{\mathrm{bias}}$ (V)')
ax2.set_ylabel('MR ratio (%)')
#ax2.grid(False)
ax2.set_ylim(0, 400 )
ax2.legend(loc='upper left', borderaxespad=0, frameon=True)

fig2.tight_layout()
plt.show()


In [ ]:
# ============================================================
# FILTER CONFIGURATION - Adjust these values as needed
# ============================================================

# Minimum I_apar threshold (A) - removes noisy low-current data
i_apar_threshold = 2e-9  # Remove data where |I_apar| < 1 nA

# Voltage cutoffs per temperature (K: max_voltage in V)
# Only temperatures listed here will have voltage limits applied
# Leave empty {} to show full voltage range for all temperatures


# Plot style configuration
plot_with_errorbars = True  # Set to False to plot without error bars (faster)
errorbar_alpha = 0.6  # Transparency of error bars (0-1)

# ============================================================


def apply_filters(data, temp, i_threshold, v_cutoffs):
    """
    Apply current and voltage filters to TMR data.
    
    Parameters:
    -----------
    data : pd.DataFrame
        DataFrame subset for a specific temperature/scan_type
    temp : int or float
        Temperature value
    i_threshold : float
        Minimum |I_apar| threshold in Amperes
    v_cutoffs : dict
        Dictionary of {temperature: max_voltage} for voltage cutoffs
    
    Returns:
    --------
    pd.DataFrame
        Filtered DataFrame
    """
    # Filter 1: Remove data where |I_apar| < threshold
    filtered = data[np.abs(data['I_apar (A)']) >= i_threshold].copy()
    
    # Filter 2: Apply voltage cutoff if specified for this temperature
    if temp in v_cutoffs:
        max_v = v_cutoffs[temp]
        filtered = filtered[np.abs(filtered['Voltage (V)']) <= max_v]
    
    return filtered


temperatures = [80, 90, 100, 110, 120, 130, 140, 150, 160]

# ── Figure 1: b_scans ────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(6, 5))

for i, temp in enumerate(temperatures):
    
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'b_scan')]
    data = apply_filters(data, temp, i_apar_threshold, v_cutoffs)
    if len(data) > 0:
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
        if plot_with_errorbars:
            ax1.errorbar(data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100,
                         yerr=data['TMR_Error'],
                         fmt='o', color=color, linewidth=1.5,
                         ecolor=color, elinewidth=0.5, capsize=0,
                         errorevery=5, alpha=errorbar_alpha,
                         label=f'{temp} K')
        else:
            ax1.plot(data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100, '-',
                     color=color, linewidth=1.5, alpha=0.7, label=f'{temp} K')

ax1.set_xlabel('$V_{\mathrm{bias}}$ (V)')
ax1.set_ylabel('MR ratio (%)')
#ax1.grid(False)
ax1.set_ylim(0, 400)
ax1.legend(loc='upper left', borderaxespad=0, frameon=True, ncol=2)

fig1.tight_layout()
plt.show()

# ── Figure 2: c_scans ────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(6, 5))

for i, temp in enumerate(temperatures):
    
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'c_scan')]
    data = apply_filters(data, temp, i_apar_threshold, v_cutoffs)
    if len(data) > 0:
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
        if plot_with_errorbars:
            ax2.errorbar( data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100,
                         yerr=data['TMR_Error'],
                         fmt='o', color=color, linewidth=1.5,
                         ecolor=color, elinewidth=0.5, capsize=0,
                         errorevery=5, alpha=errorbar_alpha,
                         label=f'{temp} K')
        else:
            ax2.plot(data['Voltage (V)'], (data['TMR_Ratio'] - 1)*100, '-',
                     color=color, linewidth=1.5, alpha=0.7, label=f'{temp} K')

ax2.set_xlabel('$V_{\mathrm{bias}}$ (V)')
ax2.set_ylabel('MR ratio (%)')
#ax2.grid(False)
ax2.set_ylim(0, 400)
ax2.legend(loc='upper left', borderaxespad=0, frameon=True, ncol=2)

fig2.tight_layout()
plt.show()


In [ ]:
# Calculate and plot both positive and negative side voltage peaks at peak TMR vs temperature
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
voltage_cutoffs = {}

peak_data_pos = []
peak_data_neg = []


if 'df_combined' in locals() or 'df_combined' in globals():
    temperatures = df_combined['temperature'].unique()
    temperatures.sort()

    for i, temp in enumerate(temperatures):
        for scan_type in ['b_scan', 'c_scan']:
            data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == scan_type)]
            if len(data) == 0:
                continue

            try:
                filtered_data = apply_filters(data, temp, i_apar_threshold, voltage_cutoffs)
            except NameError:
                filtered_data = data

            # Split data into positive and negative voltage regions
            data_pos = filtered_data[filtered_data['Voltage (V)'] > 0]
            data_neg = filtered_data[filtered_data['Voltage (V)'] < 0]

            # The peak bias is the location of a maximum, so the CSVs carry no
            # error column for it. peak_voltage_with_error fits a weighted local
            # parabola and propagates its covariance; see the markdown above.
            for branch_data, store in ((data_pos, peak_data_pos),
                                       (data_neg, peak_data_neg)):
                if len(branch_data) == 0:
                    continue
                peak = peak_voltage_with_error(branch_data['Voltage (V)'],
                                               branch_data['TMR_Ratio'],
                                               branch_data['TMR_Error'])
                if peak is None:
                    continue
                store.append({
                    'temperature': temp,
                    'scan_type': scan_type,
                    'peak_voltage': peak['peak_voltage'],
                    'peak_voltage_error': peak['peak_voltage_error'],
                    'max_tmr': peak['max_ratio'],
                    'max_tmr_error': peak['max_ratio_error'],
                    'chi2_red': peak['chi2_red'],
                    'method': peak['method'],
                    'resolved': peak['resolved'],
                })

    def peak_voltage_errors(frame):
        """Propagated peak-bias error raised to the scatter the series itself shows.

        The propagated error comes from a fit to one temperature in isolation and
        knows nothing about how well the next temperature reproduces it. The
        scatter of the series about a locally linear trend in T does, and it is
        the larger of the two over the smooth 20-90 K range.
        """
        return with_trend_floor(frame['peak_voltage_error'],
                                frame['temperature'], frame['peak_voltage'],
                                x_range=SMOOTH_T_RANGE_K)

    # Create the plot
    fig, ax = plt.subplots(figsize=(6, 5), dpi=300)

    applied_scatter = {}
    if peak_data_pos:
        df_pos = pd.DataFrame(peak_data_pos)
        b_scan_pos = df_pos[df_pos['scan_type'] == 'b_scan'].sort_values('temperature')
        c_scan_pos = df_pos[df_pos['scan_type'] == 'c_scan'].sort_values('temperature')

        if not b_scan_pos.empty:
            err, applied_scatter['b_scan V>0'] = peak_voltage_errors(b_scan_pos)
            ax.errorbar(b_scan_pos['temperature'], b_scan_pos['peak_voltage'],
                        yerr=err,
                        fmt='o-', label='b scan ($V$ > 0)', color=COLOR_B_AXIS,
                        linewidth=2, markersize=8, capsize=3)
        if not c_scan_pos.empty:
            err, applied_scatter['c_scan V>0'] = peak_voltage_errors(c_scan_pos)
            ax.errorbar(c_scan_pos['temperature'], c_scan_pos['peak_voltage'],
                        yerr=err,
                        fmt='s-', label='c scan ($V$ > 0)', color=COLOR_C_AXIS,
                        linewidth=2, markersize=8, capsize=3)

    if peak_data_neg:
        df_neg = pd.DataFrame(peak_data_neg)
        b_scan_neg = df_neg[df_neg['scan_type'] == 'b_scan'].sort_values('temperature')
        c_scan_neg = df_neg[df_neg['scan_type'] == 'c_scan'].sort_values('temperature')

        if not b_scan_neg.empty:
            err, applied_scatter['b_scan V<0'] = peak_voltage_errors(b_scan_neg)
            ax.errorbar(b_scan_neg['temperature'], b_scan_neg['peak_voltage'],
                        yerr=err,
                        fmt='v--', label='b scan ($V$ < 0)', color='#56B4E9',
                        linewidth=2, markersize=8, capsize=3)
        if not c_scan_neg.empty:
            err, applied_scatter['c_scan V<0'] = peak_voltage_errors(c_scan_neg)
            ax.errorbar(c_scan_neg['temperature'], c_scan_neg['peak_voltage'],
                        yerr=err,
                        fmt='v--', label='c scan ($V$ < 0)', color='#E69F00',
                        linewidth=2, markersize=8, capsize=3)

    ax.set_xlabel('$T$ (K)')
    ax.set_ylabel(r'$V_\mathrm{bias}^\mathrm{max(MR)}$ (V)')

    # Place legend outside to avoid obscuring data
    ax.legend(loc='best', borderaxespad=0, frameon=False)

    plt.xlim(15,165)
    plt.ylim(-0.6,0.6)
    plt.tight_layout()
    plt.show()

    # Numerical results below the figure, including which estimator was used
    summary_peaks = pd.concat([
        pd.DataFrame(peak_data_pos).assign(branch='V > 0'),
        pd.DataFrame(peak_data_neg).assign(branch='V < 0'),
    ], ignore_index=True)
    cols = ['temperature', 'scan_type', 'branch', 'peak_voltage',
            'peak_voltage_error', 'method', 'chi2_red']
    print("Peak bias voltage with uncertainty (parabola vertex, or plateau fallback):")
    print(summary_peaks.sort_values(['temperature', 'scan_type', 'branch'])[cols]
          .to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
    n_par = int((summary_peaks['method'] == 'parabola').sum())
    print(f"\nparabola fit accepted for {n_par} of {len(summary_peaks)} points; "
          f"the rest fall back to the 1-sigma plateau half-width.")
    print("\nPoint-to-point scatter of each series over "
          f"{SMOOTH_T_RANGE_K[0]:.0f}-{SMOOTH_T_RANGE_K[1]:.0f} K, "
          "applied as the floor on the plotted error bars:")
    for name, s in applied_scatter.items():
        print(f"  {name}: {1000 * s:5.1f} mV")
else:
    print("df_combined not found in namespace.")


In [ ]:
import numpy as np

# The Fowler-Nordheim ordinate is ln(|I| / V^2); the bias carries no uncertainty,
# so the whole error is the relative current error sigma_I / |I|.
I_ERROR_COL = {'I_par (A)': 'I_par_error (A)', 'I_apar (A)': 'I_apar_error (A)'}


def plot_fowler_nordheim(ax, scan_type, I_col, temperatures):
    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type) &
            (df_combined['Voltage (V)'].abs() >= 0.1)
        ].copy()
        if len(data) > 0:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            x = 1 / data['Voltage (V)']
            y = np.log(data[I_col].abs() / data['Voltage (V)']**2)
            yerr = fn_ordinate_error(data[I_col], data[I_ERROR_COL[I_col]])
            # 1/V is discontinuous across V = 0, so caps are used rather than a
            # filled band, which would bridge the gap between the two branches.
            ax.errorbar(x, y, yerr=yerr, fmt='o', color=color, ecolor=color,
                        elinewidth=0.6, capsize=0, alpha=0.7, label=f'{temp}K')
    ax.set_xlabel(r'$1/V_{\mathrm{bias}}$ (V$^{-1}$)')
    ax.set_ylabel(r'$\ln(|I|/V_{\mathrm{bias}}^2)$')
    ax.grid(False)
    ax.legend()

temperatures = [100, 90, 80, 70, 60, 50, 30, 20]

plots = [
    ('b_scan', 'I_par (A)',  'Fowler-Nordheim Plot (FM) - b_scans'),
    ('c_scan', 'I_par (A)',  'Fowler-Nordheim Plot (FM) - c_scans'),
    ('b_scan', 'I_apar (A)', 'Fowler-Nordheim Plot (AFM) - b_scans'),
    ('c_scan', 'I_apar (A)', 'Fowler-Nordheim Plot (AFM) - c_scans'),
]

for scan_type, I_col, title in plots:
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_fowler_nordheim(ax, scan_type, I_col, temperatures)
    plt.tight_layout()
    # Identify the figure in the cell output rather than with an on-figure title
    print(title)
    plt.show()

# The FN error bars are small by construction: report the size explicitly
for scan_type, I_col, title in plots:
    d = df_combined[(df_combined['scan_type'] == scan_type) &
                    (df_combined['Voltage (V)'].abs() >= 0.1) &
                    (df_combined['temperature'].isin(temperatures))]
    err = fn_ordinate_error(d[I_col], d[I_ERROR_COL[I_col]])
    print(f"{title}: median sigma[ln(|I|/V^2)] = {np.median(err):.5f}")


In [ ]:
import numpy as np

trim = 3

I_ERROR_COL = {'I_par (A)': 'I_par_error (A)', 'I_apar (A)': 'I_apar_error (A)'}


def plot_dIdV(ax, scan_type, I_col, temperatures):
    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type)
        ].copy().sort_values('Voltage (V)')
        if len(data) > 0:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            voltage = data['Voltage (V)'].values
            # np.gradient is linear, so the measured current error propagates
            # through it exactly (scripts/iv_error_propagation.py).
            dIdV, dIdV_err = didv_with_error(voltage,
                                             data[I_col].values * 1e6,
                                             data[I_ERROR_COL[I_col]].values * 1e6)
            v = voltage[trim:-trim]
            y = dIdV[trim:-trim]
            yerr = dIdV_err[trim:-trim]
            ax.plot(v, y, 'o', color=color, linewidth=1.5, alpha=0.7, label=f'{temp}K')
            # Dense curve: +-1 sigma band instead of caps.
            fill_sigma_band(ax, v, y, yerr, color=color, alpha=0.25)
    ax.set_xlabel(r'$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$dI/dV$ (μA/V)')
    ax.grid(False)
    ax.legend(ncol=2)

temps_afm = [20, 30, 40, 50, 60, 70, 80, 90, 100]
temps_fm  = [20, 30, 50, 60, 70, 80, 90, 100]

plots = [
    ('b_scan', 'I_apar (A)', 'Raw dI/dV (AFM) - b_scans', temps_afm),
    ('c_scan', 'I_apar (A)', 'Raw dI/dV (AFM) - c_scans', temps_afm),
    ('b_scan', 'I_par (A)',  'Raw dI/dV (FM) - b_scans',  temps_fm),
    ('c_scan', 'I_par (A)',  'Raw dI/dV (FM) - c_scans',  temps_fm),
]

for scan_type, I_col, title, temperatures in plots:
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_dIdV(ax, scan_type, I_col, temperatures)
    plt.tight_layout()
    # Identify the figure in the cell output rather than with an on-figure title
    print(title)
    plt.show()


In [ ]:
import numpy as np
import pandas as pd

trim = 3

# Convert peak_data to DataFrames for easy lookup
df_pos = pd.DataFrame(peak_data_pos) if peak_data_pos else pd.DataFrame()

I_ERROR_COL = {'I_par (A)': 'I_par_error (A)', 'I_apar (A)': 'I_apar_error (A)'}

def compute_normalized_dIdV(data, I_col):
    """Feenstra-normalised conductance with its propagated one-sigma error."""
    data = data.copy().sort_values('Voltage (V)')
    V = data['Voltage (V)'].values
    I = data[I_col].values
    I_err = data[I_ERROR_COL[I_col]].values
    normalized, norm_err = normalized_didv_with_error(V, I, I_err)
    V = V[trim:-trim]
    normalized, norm_err = normalized[trim:-trim], norm_err[trim:-trim]
    mask = np.abs(V) > 0.05
    return V[mask], normalized[mask], norm_err[mask]

def get_normalized_didv_at_voltage(data, I_col, peak_voltage):
    V_masked, normalized, norm_err = compute_normalized_dIdV(data, I_col)
    idx = np.argmin(np.abs(V_masked - peak_voltage))
    return V_masked[idx], normalized[idx], norm_err[idx]

def add_peak_markers(ax, scan_type, I_col, df_pos, temperatures_set):
    for _, row in df_pos[df_pos['scan_type'] == scan_type].iterrows():
        temp = row['temperature']
        if temp not in temperatures_set:
            continue
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type)
        ].copy()
        if len(data) == 0:
            continue
        i = temperatures.index(temp)
        color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
        V_pt, y_pt, y_err = get_normalized_didv_at_voltage(data, I_col, row['peak_voltage'])
        # Horizontal bar = uncertainty on the peak-MR bias, vertical = propagated
        # error of the normalised conductance at that bias.
        ax.errorbar(V_pt, y_pt, xerr=row['peak_voltage_error'], yerr=y_err,
                    fmt='o', markersize=10, markerfacecolor=color,
                    markeredgecolor='#000000', markeredgewidth=1.2,
                    ecolor='#000000', elinewidth=1.0, capsize=3, zorder=6)

temperatures = [20, 30, 40, 50, 60, 70, 80, 90, 100]
temperatures_set = set(temperatures)

plots = [
    ('b_scan', 'I_apar (A)', 'Normalized dI/dV (AFM) — b-axis', False),
    ('c_scan', 'I_apar (A)', 'Normalized dI/dV (AFM) — c-axis', False),
    ('b_scan', 'I_par (A)',  'Normalized dI/dV (FM) — b-axis',  True),
    ('c_scan', 'I_par (A)',  'Normalized dI/dV (FM) — c-axis',  True),
]

for scan_type, I_col, title, add_markers in plots:
    fig, ax = plt.subplots(figsize=(6, 5))

    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type)
        ].copy().sort_values('Voltage (V)')
        if len(data) > 0:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            V_masked, normalized, norm_err = compute_normalized_dIdV(data, I_col)
            ax.plot(V_masked, normalized, 'o', color=color, alpha=0.7, label=f'{temp} K')
            # Dense curve: +-1 sigma band rather than caps on every point.
            fill_sigma_band(ax, V_masked, normalized, norm_err,
                            color=color, alpha=0.25)

    if add_markers:
        add_peak_markers(ax, scan_type, I_col, df_pos, temperatures_set)

    ax.set_xlabel(r'$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$(dI/dV)/(I/V)$')
    ax.set_ylim(1, 4.25)
    ax.grid(False)
    ax.legend(loc='upper left', borderaxespad=0, ncol=2, frameon=True)
    fig.tight_layout()
    # Identify the figure in the cell output rather than with an on-figure title
    print(title)
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

TEMP = 20
trim = 2

I_ERROR_COL = {'I_par (A)': 'I_par_error (A)', 'I_apar (A)': 'I_apar_error (A)'}

def compute_normalized_didv(data, I_col):
    """Feenstra-normalised conductance with its propagated one-sigma error."""
    data = data.copy().sort_values('Voltage (V)')
    V = data['Voltage (V)'].values
    I = data[I_col].values
    I_err = data[I_ERROR_COL[I_col]].values
    normalized, norm_err = normalized_didv_with_error(V, I, I_err)
    V = V[trim:-trim]
    normalized, norm_err = normalized[trim:-trim], norm_err[trim:-trim]
    mask = np.abs(V) > 0.05
    return V[mask], normalized[mask], norm_err[mask]


def plot_axis(ax, scan_type, title):
    data = df_combined[
        (df_combined['temperature'] == TEMP) &
        (df_combined['scan_type'] == scan_type)
    ].copy()

    if len(data) == 0:
        print(f'{title} — no data at {TEMP} K')
        return

    V_afm, norm_afm, err_afm = compute_normalized_didv(data, 'I_apar (A)')
    ax.plot(V_afm, norm_afm, 'o', color=OKABE_ITO_CYCLE[0], alpha=0.85,
            markersize=4, linewidth=1.4, label='AFM')
    # Dense curve: +-1 sigma band instead of a cap on every point.
    fill_sigma_band(ax, V_afm, norm_afm, err_afm,
                    color=OKABE_ITO_CYCLE[0], alpha=0.25)

    V_fm, norm_fm, err_fm = compute_normalized_didv(data, 'I_par (A)')
    ax.plot(V_fm, norm_fm, 'o', color=OKABE_ITO_CYCLE[1], alpha=0.85,
            markersize=4, linewidth=1.4, label='FM')
    fill_sigma_band(ax, V_fm, norm_fm, err_fm,
                    color=OKABE_ITO_CYCLE[1], alpha=0.25)

    peaks = {}
    for norm, V, label in [(norm_afm, V_afm, 'AFM'), (norm_fm, V_fm, 'FM')]:
        mask = (V >= 0.5) & (V <= 0.9)
        if mask.any():
            peak_V = V[mask][np.argmax(norm[mask])]
            peaks[label] = peak_V
            ax.axvline(peak_V, color='black', linewidth=2.0,
                       linestyle='--', alpha=0.9, zorder=5)

    if 'AFM' in peaks and 'FM' in peaks:
        V_afm_peak = peaks['AFM']
        V_fm_peak  = peaks['FM']
        delta_V    = abs(V_fm_peak - V_afm_peak)
        arrow_y    = 1.5
        mid_V      = (V_afm_peak + V_fm_peak) / 2

        ax.annotate('', xy=(V_fm_peak, arrow_y), xytext=(V_afm_peak, arrow_y),
                    arrowprops=dict(arrowstyle='<->', color='black',
                                   lw=1.5, mutation_scale=14))
        ax.text(mid_V, arrow_y + 1.15,
                f'$\\Delta V = {delta_V*1000:.0f}$ mV',
                ha='center', va='top', color='black', fontsize=14,
                rotation=90)

    ax.set_xlabel(r'$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$(dI/dV)/(I/V)$')
    ax.set_xlim(-1, 1)
    ax.set_ylim(1, 4.25)
    ax.grid(False)
    # Shaded bands have no legend entry of their own (fill_between takes no
    # label); add one proxy swatch so the plot states what the shading means.
    handles, labels = ax.get_legend_handles_labels()
    handles.append(Patch(facecolor='0.6', alpha=0.3, label=r'$\pm1\sigma$'))
    ax.legend(handles=handles)


for scan_type, title in [('b_scan', 'b-axis'), ('c_scan', 'c-axis')]:
    fig, ax = plt.subplots(figsize=(6, 5))
    plot_axis(ax, scan_type=scan_type, title=title)
    fig.tight_layout()
    # Identify the figure in the cell output rather than with an on-figure title
    print(f'Normalized dI/dV at {TEMP} K — {title}')
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# --- Configuration ---
temperatures = [20, 30, 40, 50, 60, 70, 80, 90, 100]

# --- Figure 1: d²I/dV² — AFM state ---
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for scan_type, ax in zip(['b_scan', 'c_scan'], [ax1, ax2]):
    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type)
        ].copy().sort_values('Voltage (V)')

        if len(data) > 2:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            V = data['Voltage (V)'].values
            I = data['I_apar (A)'].values
            I_err = data['I_apar_error (A)'].values

            # --- Calculation: d²I/dV² with the current error propagated through
            # both differentiations (the 1/spacing amplification is squared, so
            # these error bars are large by construction).
            d2IdV2, d2IdV2_err = d2idv2_with_error(V, I, I_err)

            # --- Updated Masking Mechanism (0.05 < |V| < 0.9) ---
            mask = (np.abs(V) > 0.01) & (np.abs(V) < 0.9)

            ax.plot(V[mask], d2IdV2[mask], 'o', color=color, alpha=0.7, markersize=3, label=f'{temp}K')
            # Differentiating twice amplifies the current error by ~1/h^2, so
            # this band is wide and is kept faint to leave the data legible.
            fill_sigma_band(ax, V[mask], d2IdV2[mask], d2IdV2_err[mask],
                            color=color, alpha=0.10)

    ax.set_xlabel(r'$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$(d^2I/dV^2)$ (A/V$^2$)')
    ax.grid(False)
    # Shaded bands have no legend entry of their own (fill_between takes no
    # label); add one proxy swatch so the plot states what the shading means.
    handles, labels = ax.get_legend_handles_labels()
    handles.append(Patch(facecolor='0.6', alpha=0.3, label=r'$\pm1\sigma$'))
    ax.legend(handles=handles, ncol=2)
    ax.set_ylim(-1E-5, 1E-5)

fig1.tight_layout()
# Identify the figure in the cell output rather than with on-figure titles
print('d2I/dV2 (AFM) — left: b_scan, right: c_scan')
plt.show()

# --- Figure 2: d²I/dV² — FM state ---
fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(16, 6))

for scan_type, ax in zip(['b_scan', 'c_scan'], [ax3, ax4]):
    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type)
        ].copy().sort_values('Voltage (V)')

        if len(data) > 2:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            V = data['Voltage (V)'].values
            I = data['I_par (A)'].values
            I_err = data['I_par_error (A)'].values

            # --- Calculation: d²I/dV² ---
            d2IdV2, d2IdV2_err = d2idv2_with_error(V, I, I_err)

            # --- Updated Masking Mechanism (0.05 < |V| < 0.9) ---
            mask = (np.abs(V) > 0.01) & (np.abs(V) < 0.9)

            ax.plot(V[mask], d2IdV2[mask], 'o', color=color, alpha=0.7, markersize=3, label=f'{temp}K')
            # Differentiating twice amplifies the current error by ~1/h^2, so
            # this band is wide and is kept faint to leave the data legible.
            fill_sigma_band(ax, V[mask], d2IdV2[mask], d2IdV2_err[mask],
                            color=color, alpha=0.10)

    ax.set_xlabel(r'$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$(d^2I/dV^2)$ (A/V$^2$)')
    ax.grid(False)
    # Shaded bands have no legend entry of their own (fill_between takes no
    # label); add one proxy swatch so the plot states what the shading means.
    handles, labels = ax.get_legend_handles_labels()
    handles.append(Patch(facecolor='0.6', alpha=0.3, label=r'$\pm1\sigma$'))
    ax.legend(handles=handles, ncol=2)
    ax.set_ylim(-1E-5, 1E-5)

fig2.tight_layout()
print('d2I/dV2 (FM) — left: b_scan, right: c_scan')
plt.show()

# The propagated second-derivative error, in the units of the panels above
for state, I_col, err_col in (('AFM', 'I_apar (A)', 'I_apar_error (A)'),
                              ('FM', 'I_par (A)', 'I_par_error (A)')):
    for scan_type in ('b_scan', 'c_scan'):
        sizes = []
        for temp in temperatures:
            d = df_combined[(df_combined['temperature'] == temp) &
                            (df_combined['scan_type'] == scan_type)
                            ].sort_values('Voltage (V)')
            if len(d) < 5:
                continue
            _, err = d2idv2_with_error(d['Voltage (V)'].values,
                                       d[I_col].values, d[err_col].values)
            sizes.append(np.median(err))
        if sizes:
            print(f"{state} {scan_type}: median sigma(d2I/dV2) over "
                  f"{len(sizes)} temperatures = {np.median(sizes):.2e} A/V^2 "
                  f"(range {min(sizes):.1e} to {max(sizes):.1e}); "
                  f"panel spans +-1e-05 A/V^2")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
import pandas as pd

# Convert peak_data to DataFrames for easy lookup
df_pos = pd.DataFrame(peak_data_pos) if peak_data_pos else pd.DataFrame()
df_neg = pd.DataFrame(peak_data_neg) if peak_data_neg else pd.DataFrame()

TEMP = 50  # K — single temperature of interest

I_ERROR_COL = {'I_par (A)': 'I_par_error (A)', 'I_apar (A)': 'I_apar_error (A)'}

def compute_normalized_didv(data, I_col):
    """Return (V_masked, normalized dI/dV, one-sigma error) for one temperature/scan."""
    data = data.copy().sort_values('Voltage (V)')
    V = data['Voltage (V)'].values
    I = data[I_col].values
    I_err = data[I_ERROR_COL[I_col]].values
    normalized, norm_err = normalized_didv_with_error(V, I, I_err)
    mask = np.abs(V) > 0.05
    return V[mask], normalized[mask], norm_err[mask]


def get_normalized_didv_at_voltage(data, I_col, peak_voltage):
    """Return (V, normalized dI/dV, error) at the point closest to peak_voltage."""
    V_masked, normalized, norm_err = compute_normalized_didv(data, I_col)
    idx = np.argmin(np.abs(V_masked - peak_voltage))
    return V_masked[idx], normalized[idx], norm_err[idx]


def add_peak_markers(ax, scan_type, I_col, df_pos, df_neg):
    """Overlay star markers at peak TMR voltages on a normalized dI/dV axis."""
    for df in [df_pos, df_neg]:
        if df.empty:
            continue
        subset = df[(df['scan_type'] == scan_type) & (df['temperature'] == TEMP)]
        for _, row in subset.iterrows():
            data = df_combined[
                (df_combined['temperature'] == TEMP) &
                (df_combined['scan_type'] == scan_type)
            ].copy()
            if len(data) == 0:
                continue
            V_pt, y_pt, y_err = get_normalized_didv_at_voltage(data, I_col, row['peak_voltage'])
            # Horizontal bar = uncertainty on the peak-MR bias from the weighted
            # parabola fit; vertical = propagated error of the normalised signal.
            ax.errorbar(V_pt, y_pt, xerr=row['peak_voltage_error'], yerr=y_err,
                        fmt='*', markersize=16, markerfacecolor='white',
                        markeredgecolor='#D55E00', markeredgewidth=1.2,
                        ecolor='#D55E00', elinewidth=1.0, capsize=3, zorder=6,
                        label='Peak MR' if _ == subset.index[0] else '')


def plot_axis(ax, scan_type, title):
    """
    Plot AFM (I_apar) and FM (I_par) normalized dI/dV at TEMP K on a single axes.

    Visual encoding
    ---------------
    State   Color       Line style
    AFM     #0072B2     dashed  (--)
    FM      #009E73     solid   (-)
    """
    data = df_combined[
        (df_combined['temperature'] == TEMP) &
        (df_combined['scan_type'] == scan_type)
    ].copy()

    if len(data) == 0:
        print(f'{title} — no data at {TEMP} K')
        return

    # --- AFM ---
    V_afm, norm_afm, err_afm = compute_normalized_didv(data, 'I_apar (A)')
    ax.plot(V_afm, norm_afm, 'o', color='#0072B2', alpha=0.85,
            markersize=4, linewidth=1.4, label='AFM')
    # Dense curve: +-1 sigma band instead of a cap on every point.
    fill_sigma_band(ax, V_afm, norm_afm, err_afm, color='#0072B2', alpha=0.25)

    # --- FM ---
    V_fm, norm_fm, err_fm = compute_normalized_didv(data, 'I_par (A)')
    ax.plot(V_fm, norm_fm, 'o', color='#009E73', alpha=0.85,
            markersize=4, linewidth=1.4, label='FM')
    fill_sigma_band(ax, V_fm, norm_fm, err_fm, color='#009E73', alpha=0.25)

    # --- Peak markers on FM ---
    add_peak_markers(ax, scan_type, 'I_par (A)', df_pos, df_neg)

    ax.set_xlabel(r'$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$(dI/dV)/(I/V)$')
    ax.set_ylim(1, 4.25)
    ax.grid(False)
    # Shaded bands have no legend entry of their own (fill_between takes no
    # label); add one proxy swatch so the plot states what the shading means.
    handles, labels = ax.get_legend_handles_labels()
    handles.append(Patch(facecolor='0.6', alpha=0.3, label=r'$\pm1\sigma$'))
    ax.legend(handles=handles)


# --- Figure 1: b-axis ---
fig1, ax1 = plt.subplots(figsize=(7, 5))
plot_axis(ax1, scan_type='b_scan', title=f'Normalized dI/dV — b-axis, {TEMP} K')
fig1.tight_layout()
# Identify the figure in the cell output rather than with an on-figure title
print(f'Normalized dI/dV — b-axis, {TEMP} K')
plt.show()

# --- Figure 2: c-axis ---
fig2, ax2 = plt.subplots(figsize=(7, 5))
plot_axis(ax2, scan_type='c_scan', title=f'Normalized dI/dV — c-axis, {TEMP} K')
fig2.tight_layout()
print(f'Normalized dI/dV — c-axis, {TEMP} K')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
import pandas as pd

# Convert peak_data to DataFrames for easy lookup
df_pos = pd.DataFrame(peak_data_pos) if peak_data_pos else pd.DataFrame()
df_neg = pd.DataFrame(peak_data_neg) if peak_data_neg else pd.DataFrame()

TEMP = 70  # K — single temperature of interest

I_ERROR_COL = {'I_par (A)': 'I_par_error (A)', 'I_apar (A)': 'I_apar_error (A)'}

def compute_normalized_didv(data, I_col):
    """Return (V_masked, normalized dI/dV, one-sigma error) for one temperature/scan."""
    data = data.copy().sort_values('Voltage (V)')
    V = data['Voltage (V)'].values
    I = data[I_col].values
    I_err = data[I_ERROR_COL[I_col]].values
    normalized, norm_err = normalized_didv_with_error(V, I, I_err)
    mask = np.abs(V) > 0.05
    return V[mask], normalized[mask], norm_err[mask]


def get_normalized_didv_at_voltage(data, I_col, peak_voltage):
    """Return (V, normalized dI/dV, error) at the point closest to peak_voltage."""
    V_masked, normalized, norm_err = compute_normalized_didv(data, I_col)
    idx = np.argmin(np.abs(V_masked - peak_voltage))
    return V_masked[idx], normalized[idx], norm_err[idx]


def add_peak_markers(ax, scan_type, I_col, df_pos, df_neg):
    """Overlay star markers at peak TMR voltages on a normalized dI/dV axis."""
    for df in [df_pos, df_neg]:
        if df.empty:
            continue
        subset = df[(df['scan_type'] == scan_type) & (df['temperature'] == TEMP)]
        for _, row in subset.iterrows():
            data = df_combined[
                (df_combined['temperature'] == TEMP) &
                (df_combined['scan_type'] == scan_type)
            ].copy()
            if len(data) == 0:
                continue
            V_pt, y_pt, y_err = get_normalized_didv_at_voltage(data, I_col, row['peak_voltage'])
            # Horizontal bar = uncertainty on the peak-MR bias from the weighted
            # parabola fit; vertical = propagated error of the normalised signal.
            ax.errorbar(V_pt, y_pt, xerr=row['peak_voltage_error'], yerr=y_err,
                        fmt='*', markersize=16, markerfacecolor='white',
                        markeredgecolor='#D55E00', markeredgewidth=1.2,
                        ecolor='#D55E00', elinewidth=1.0, capsize=3, zorder=6,
                        label='Peak MR' if _ == subset.index[0] else '')


def plot_axis(ax, scan_type, title):
    """
    Plot AFM (I_apar) and FM (I_par) normalized dI/dV at TEMP K on a single axes.

    Visual encoding
    ---------------
    State   Color       Line style
    AFM     #0072B2     dashed  (--)
    FM      #009E73     solid   (-)
    """
    data = df_combined[
        (df_combined['temperature'] == TEMP) &
        (df_combined['scan_type'] == scan_type)
    ].copy()

    if len(data) == 0:
        print(f'{title} — no data at {TEMP} K')
        return

    # --- AFM ---
    V_afm, norm_afm, err_afm = compute_normalized_didv(data, 'I_apar (A)')
    ax.plot(V_afm, norm_afm, 'o', color='#0072B2', alpha=0.85,
            markersize=4, linewidth=1.4, label='AFM')
    # Dense curve: +-1 sigma band instead of a cap on every point.
    fill_sigma_band(ax, V_afm, norm_afm, err_afm, color='#0072B2', alpha=0.25)

    # --- FM ---
    V_fm, norm_fm, err_fm = compute_normalized_didv(data, 'I_par (A)')
    ax.plot(V_fm, norm_fm, 'o', color='#009E73', alpha=0.85,
            markersize=4, linewidth=1.4, label='FM')
    fill_sigma_band(ax, V_fm, norm_fm, err_fm, color='#009E73', alpha=0.25)

    # --- Peak markers on FM ---
    add_peak_markers(ax, scan_type, 'I_par (A)', df_pos, df_neg)

    ax.set_xlabel(r'$V_{\mathrm{bias}}$ (V)')
    ax.set_ylabel('$(dI/dV)/(I/V)$')
    ax.set_ylim(1, 4.25)
    ax.grid(False)
    # Shaded bands have no legend entry of their own (fill_between takes no
    # label); add one proxy swatch so the plot states what the shading means.
    handles, labels = ax.get_legend_handles_labels()
    handles.append(Patch(facecolor='0.6', alpha=0.3, label=r'$\pm1\sigma$'))
    ax.legend(handles=handles)


# --- Figure 1: b-axis ---
fig1, ax1 = plt.subplots(figsize=(7, 5))
plot_axis(ax1, scan_type='b_scan', title=f'Normalized dI/dV — b-axis, {TEMP} K')
fig1.tight_layout()
# Identify the figure in the cell output rather than with an on-figure title
print(f'Normalized dI/dV — b-axis, {TEMP} K')
plt.show()

# --- Figure 2: c-axis ---
fig2, ax2 = plt.subplots(figsize=(7, 5))
plot_axis(ax2, scan_type='c_scan', title=f'Normalized dI/dV — c-axis, {TEMP} K')
fig2.tight_layout()
print(f'Normalized dI/dV — c-axis, {TEMP} K')
plt.show()


## 9. I_par at Fixed Voltage vs Temperature

Analyze how the parallel current (I_par) varies with temperature at a fixed bias voltage of 0.5V.

In [ ]:
# Extract I_par at V=0.25V for each temperature and scan type
target_voltage = 0.1  # V
tolerance = 0.01  # Voltage tolerance for matching

def get_current_at_voltage(group, v_target, tolerance=0.01):
    """Extract current closest to target voltage."""
    voltage_diff = np.abs(group['Voltage (V)'] - v_target)
    closest_idx = voltage_diff.idxmin()
    
    if voltage_diff.loc[closest_idx] < tolerance:
        return pd.Series({
            'I_par': group.loc[closest_idx, 'I_par (A)'],
            'I_par_error': group.loc[closest_idx, 'I_par_error (A)'],
            'voltage_actual': group.loc[closest_idx, 'Voltage (V)'],
            'I_apar': group.loc[closest_idx, 'I_apar (A)'],
            'I_apar_error': group.loc[closest_idx, 'I_apar_error (A)']
        })
    else:
        return pd.Series({
            'I_par': np.nan,
            'I_par_error': np.nan,
            'voltage_actual': np.nan,
            'I_apar': np.nan,
            'I_apar_error': np.nan
        })

# Apply to each temperature/scan_type group
i_par_vs_temp = df_combined.groupby(['temperature', 'scan_type']).apply(
    lambda g: get_current_at_voltage(g, target_voltage, tolerance)
).reset_index()

# Separate b_scan and c_scan
b_scan_data = i_par_vs_temp[i_par_vs_temp['scan_type'] == 'b_scan'].sort_values('temperature')
c_scan_data = i_par_vs_temp[i_par_vs_temp['scan_type'] == 'c_scan'].sort_values('temperature')

print(f"I_par at {target_voltage}V vs Temperature")
print(f"\nb_scans:")
print(b_scan_data)
print(f"\nc_scans:")
print(c_scan_data)

In [ ]:
# Plot I_par at 0.5V vs Temperature for b and c-axis
fig, ax = plt.subplots(figsize=(6, 5), dpi = 300)

# I_par_error/I_apar_error are standard errors of the mean over field points that
# are correlated within a magnetic state (scripts/mr_analysis.py) — the same class
# of underestimated error as TMR_Error elsewhere in this notebook. Raise each to
# the point-to-point scatter the series itself shows before plotting.
scatter_28 = {}

# Plot b_scans
err, scatter_28['b_scan I_par'] = with_trend_floor(
    b_scan_data['I_par_error'], b_scan_data['temperature'], b_scan_data['I_par'],
    x_range=SMOOTH_T_RANGE_K)
ax.errorbar(b_scan_data['temperature'], b_scan_data['I_par'],
            yerr=err,
            fmt='o-', color=OKABE_ITO_CYCLE[0], markersize=8, linewidth=2,
            capsize=5, capthick=2, label='b-axis, FM', alpha=0.8)

# Plot c_scans
err, scatter_28['c_scan I_par'] = with_trend_floor(
    c_scan_data['I_par_error'], c_scan_data['temperature'], c_scan_data['I_par'],
    x_range=SMOOTH_T_RANGE_K)
ax.errorbar(c_scan_data['temperature'], c_scan_data['I_par'],
            yerr=err,
            fmt='s-', color=OKABE_ITO_CYCLE[1], markersize=8, linewidth=2,
            capsize=5, capthick=2, label='c-axis,FM', alpha=0.8)
# Plot b_scans
err, scatter_28['b_scan I_apar'] = with_trend_floor(
    b_scan_data['I_apar_error'], b_scan_data['temperature'], b_scan_data['I_apar'],
    x_range=SMOOTH_T_RANGE_K)
ax.errorbar(b_scan_data['temperature'], b_scan_data['I_apar'],
            yerr=err,
            fmt='o-', color=OKABE_ITO_CYCLE[2], markersize=8, linewidth=2,
            capsize=5, capthick=2, label='AFM', alpha=0.8)


ax.set_yscale('log')

ax.set_xlabel('$T$ (K)')
ax.set_ylabel('$I$ (A)')
ax.grid(False, alpha=0.3, linestyle='--')
ax.legend(loc='best')
ax.set_xlim(20,160)
ax.set_ylim(5E-10,4E-7)

plt.tight_layout()
plt.show()

# Error-bar magnitudes behind the figure above
print(f"Currents at V = {target_voltage} V, with the measured one-sigma errors:")
for name, frame in (('b_scan', b_scan_data), ('c_scan', c_scan_data)):
    rel_fm = 100 * (frame['I_par_error'] / frame['I_par']).abs()
    rel_afm = 100 * (frame['I_apar_error'] / frame['I_apar']).abs()
    print(f"  {name}: median relative error  FM {rel_fm.median():.2f} %, "
          f"AFM {rel_afm.median():.2f} %")
print("\nPoint-to-point scatter over "
      f"{SMOOTH_T_RANGE_K[0]:.0f}-{SMOOTH_T_RANGE_K[1]:.0f} K, applied as the "
      "floor on the plotted error bars:")
for name, s in scatter_28.items():
    print(f"  {name}: {s:.3e} A")


In [ ]:
# Plot I_par at 0.5V vs Temperature for b and c-axis
fig, ax = plt.subplots(figsize=(6, 5), dpi = 300)

# Same scatter-floor correction as the cell above.
scatter_29 = {}

# Plot b_scans
err, scatter_29['b_scan I_par'] = with_trend_floor(
    b_scan_data['I_par_error'], b_scan_data['temperature'], b_scan_data['I_par'],
    x_range=SMOOTH_T_RANGE_K)
ax.errorbar(b_scan_data['temperature'], b_scan_data['I_par'],
            yerr=err,
            fmt='o-', color=OKABE_ITO_CYCLE[0], markersize=8, linewidth=2,
            capsize=5, capthick=2, label='b-axis, FM', alpha=0.8)

# Plot c_scans
err, scatter_29['c_scan I_par'] = with_trend_floor(
    c_scan_data['I_par_error'], c_scan_data['temperature'], c_scan_data['I_par'],
    x_range=SMOOTH_T_RANGE_K)
ax.errorbar(c_scan_data['temperature'], c_scan_data['I_par'],
            yerr=err,
            fmt='s-', color=OKABE_ITO_CYCLE[1], markersize=8, linewidth=2,
            capsize=5, capthick=2, label='c-axis,FM', alpha=0.8)



ax.set_yscale('log')

ax.set_xlabel('$T$ (K)')
ax.set_ylabel('$I$ (A)')
ax.grid(False, alpha=0.3, linestyle='--')
ax.legend(loc='best')
ax.set_xlim(20,160)
ax.set_ylim(5E-10,4E-7)

plt.tight_layout()
plt.show()


In [ ]:
# Plot I_apar and I_par at 0.25V vs Temperature with dual y-axes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Same scatter-floor correction as the I(T) cells above, applied to all four series.
scatter_30 = {}

def _floored(frame, col, err_col, key):
    err, scatter_30[key] = with_trend_floor(
        frame[err_col], frame['temperature'], frame[col], x_range=SMOOTH_T_RANGE_K)
    return err

# Left subplot: b-axis
ax1_right = ax1.twinx()  # Create right y-axis for I_par

# Plot I_apar on left axis (convert to µA)
line1 = ax1.errorbar(b_scan_data['temperature'], b_scan_data['I_apar'] * 1e6,
                     yerr=_floored(b_scan_data, 'I_apar', 'I_apar_error', 'b_scan I_apar') * 1e6,
                     fmt='o-', color=COLOR_B_AXIS, markersize=8, linewidth=2,
                     capsize=5, capthick=2, label='I_apar', alpha=0.8)

# Plot I_par on right axis (keep in A, convert to µA for display)
line2 = ax1_right.errorbar(b_scan_data['temperature'], b_scan_data['I_par'] * 1e6,
                            yerr=_floored(b_scan_data, 'I_par', 'I_par_error', 'b_scan I_par') * 1e6,
                            fmt='s-', color='#56B4E9', markersize=8, linewidth=2,
                            capsize=5, capthick=2, label='I_par', alpha=0.8)

ax1.set_xlabel('$T$ (K)')
ax1.set_ylabel(r'$I_\mathrm{AFM}$ (µA)', color=COLOR_B_AXIS)
ax1_right.set_ylabel(r'$I_\mathrm{FM}$ (µA)', color='#56B4E9')
ax1.tick_params(axis='y', labelcolor=COLOR_B_AXIS)
ax1_right.tick_params(axis='y', labelcolor='#56B4E9')
ax1.grid(False, alpha=0.3, linestyle='--')

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_right.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# Right subplot: c-axis
ax2_right = ax2.twinx()  # Create right y-axis for I_par

# Plot I_apar on left axis (convert to µA)
line3 = ax2.errorbar(c_scan_data['temperature'], c_scan_data['I_apar'] * 1e6,
                     yerr=_floored(c_scan_data, 'I_apar', 'I_apar_error', 'c_scan I_apar') * 1e6,
                     fmt='o-', color=COLOR_C_AXIS, markersize=8, linewidth=2,
                     capsize=5, capthick=2, label='I_apar', alpha=0.8)

# Plot I_par on right axis (convert to µA for display)
line4 = ax2_right.errorbar(c_scan_data['temperature'], c_scan_data['I_par'] * 1e6,
                            yerr=_floored(c_scan_data, 'I_par', 'I_par_error', 'c_scan I_par') * 1e6,
                            fmt='s-', color='#E69F00', markersize=8, linewidth=2,
                            capsize=5, capthick=2, label='I_par', alpha=0.8)

ax2.set_xlabel('$T$ (K)')
ax2.set_ylabel(r'$I_\mathrm{AFM}$ (µA)', color=COLOR_C_AXIS)
ax2_right.set_ylabel(r'$I_\mathrm{FM}$ (µA)', color='#E69F00')
ax2.tick_params(axis='y', labelcolor=COLOR_C_AXIS)
ax2_right.tick_params(axis='y', labelcolor='#E69F00')
ax2.grid(False, alpha=0.3, linestyle='--')

# Combine legends
lines3, labels3 = ax2.get_legend_handles_labels()
lines4, labels4 = ax2_right.get_legend_handles_labels()
ax2.legend(lines3 + lines4, labels3 + labels4, loc='upper left')

plt.tight_layout()
# Identify the figure in the cell output rather than with on-figure titles
print(f'I_apar and I_par at {target_voltage} V vs temperature — left: b-axis, right: c-axis')
plt.show()


In [ ]:
# Compare I_par at 0.25V: b-axis vs c-axis
fig, ax = plt.subplots(figsize=(12, 7))

# Same scatter-floor correction as the I(T) cells above.
err_b, scatter_b = with_trend_floor(
    b_scan_data['I_par_error'], b_scan_data['temperature'], b_scan_data['I_par'],
    x_range=SMOOTH_T_RANGE_K)
err_c, scatter_c = with_trend_floor(
    c_scan_data['I_par_error'], c_scan_data['temperature'], c_scan_data['I_par'],
    x_range=SMOOTH_T_RANGE_K)

# Plot b-axis I_par
ax.errorbar(b_scan_data['temperature'], b_scan_data['I_par'] * 1e6,
            yerr=err_b * 1e6,
            fmt='o-', color=COLOR_B_AXIS, markersize=8, linewidth=2,
            capsize=5, capthick=2, label='b-axis', alpha=0.8)

# Plot c-axis I_par
ax.errorbar(c_scan_data['temperature'], c_scan_data['I_par'] * 1e6,
            yerr=err_c * 1e6,
            fmt='s-', color=COLOR_C_AXIS, markersize=8, linewidth=2,
            capsize=5, capthick=2, label='c-axis', alpha=0.8)

ax.set_xlabel('$T$ (K)')
ax.set_ylabel(r'$I_\mathrm{FM}$ (µA)')
ax.grid(False, alpha=0.3, linestyle='--')
ax.legend(loc='best') # Set lower limit for log scale
plt.tight_layout()
# Identify the figure in the cell output rather than with an on-figure title
print(f'Parallel current I_par at {target_voltage} V vs temperature: b-axis vs c-axis')
plt.show()


In [ ]:
# Compare I_apar at 0.25V: b-axis vs c-axis
fig, ax = plt.subplots(figsize=(12, 7))

# Same scatter-floor correction as the I(T) cells above.
err_b, scatter_b = with_trend_floor(
    b_scan_data['I_apar_error'], b_scan_data['temperature'], b_scan_data['I_apar'],
    x_range=SMOOTH_T_RANGE_K)
err_c, scatter_c = with_trend_floor(
    c_scan_data['I_apar_error'], c_scan_data['temperature'], c_scan_data['I_apar'],
    x_range=SMOOTH_T_RANGE_K)

# Plot b-axis I_apar
ax.errorbar(b_scan_data['temperature'], b_scan_data['I_apar'] * 1e6,
            yerr=err_b * 1e6,
            fmt='o-', color=COLOR_B_AXIS, markersize=8, linewidth=2,
            capsize=5, capthick=2, label='b-axis', alpha=0.8)

# Plot c-axis I_apar
ax.errorbar(c_scan_data['temperature'], c_scan_data['I_apar'] * 1e6,
            yerr=err_c * 1e6,
            fmt='s-', color=COLOR_C_AXIS, markersize=8, linewidth=2,
            capsize=5, capthick=2, label='c-axis', alpha=0.8)

ax.set_xlabel('$T$ (K)')
ax.set_ylabel(r'$I_\mathrm{AFM}$ (µA)')
ax.grid(False, alpha=0.3, linestyle='--')
ax.legend(loc='best')
plt.tight_layout()
# Identify the figure in the cell output rather than with an on-figure title
print(f'Anti-parallel current I_apar at {target_voltage} V vs temperature: b-axis vs c-axis')
plt.show()


In [ ]:
# Calculate and plot voltage at peak TMR vs temperature
import pandas as pd
import matplotlib.pyplot as plt

peak_tmr_data = []

# Assuming df_combined and apply_filters are in the namespace
if 'df_combined' in locals() or 'df_combined' in globals():
    temperatures = df_combined['temperature'].unique()
    temperatures.sort()

    for i, temp in enumerate(temperatures):
        for scan_type in ['b_scan', 'c_scan']:
            data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == scan_type)]
            if len(data) == 0:
                continue

            # Use 'i_apar_threshold' and 'voltage_cutoffs' if defined
            try:
                filtered_data = apply_filters(data, temp, i_apar_threshold, voltage_cutoffs)
            except NameError:
                # Fallback if filters are not available in the current namespace
                filtered_data = data

            if len(filtered_data) > 0:
                # The peak bias is the location of a maximum, so it has no error
                # column in the CSVs. peak_voltage_with_error fits a weighted
                # local parabola around the argmax and propagates its covariance
                # to the vertex; where that fit is not valid it falls back to the
                # 1-sigma plateau half-width. See the markdown in section 6.
                peak = peak_voltage_with_error(filtered_data['Voltage (V)'],
                                               filtered_data['TMR_Ratio'],
                                               filtered_data['TMR_Error'])
                if peak is None:
                    continue
                peak_tmr_data.append({
                    'temperature': temp,
                    'scan_type': scan_type,
                    'peak_voltage': peak['peak_voltage'],
                    'peak_voltage_error': peak['peak_voltage_error'],
                    'max_tmr': peak['max_ratio'],
                    'max_tmr_error': peak['max_ratio_error'],
                    'chi2_red': peak['chi2_red'],
                    'method': peak['method'],
                    'resolved': peak['resolved'],
                })

    if peak_tmr_data:
        df_peak_tmr = pd.DataFrame(peak_tmr_data).sort_values('temperature')

        # Create the plot
        fig, ax = plt.subplots(figsize=(6, 5), dpi=300)

        # Both series carry the same two-part uncertainty: the propagated error
        # from the local parabola fit at one temperature, raised to the scatter the
        # series shows from one temperature to the next. The propagated error alone
        # describes a single fit and cannot see that reproducibility.
        peak_scatter = {}

        # Plot b_scan
        b_scan_peaks = df_peak_tmr[df_peak_tmr['scan_type'] == 'b_scan']
        if not b_scan_peaks.empty:
            err_b, peak_scatter['b_scan'] = with_trend_floor(
                b_scan_peaks['peak_voltage_error'], b_scan_peaks['temperature'],
                b_scan_peaks['peak_voltage'], x_range=SMOOTH_T_RANGE_K)
            ax.errorbar(b_scan_peaks['temperature'], b_scan_peaks['peak_voltage'],
                        yerr=err_b,
                        fmt='o-', label='b_scan', color=COLOR_B_AXIS,
                        linewidth=2, markersize=8, capsize=3)

        # Plot c_scan
        c_scan_peaks = df_peak_tmr[df_peak_tmr['scan_type'] == 'c_scan']
        if not c_scan_peaks.empty:
            err_c, peak_scatter['c_scan'] = with_trend_floor(
                c_scan_peaks['peak_voltage_error'], c_scan_peaks['temperature'],
                c_scan_peaks['peak_voltage'], x_range=SMOOTH_T_RANGE_K)
            ax.errorbar(c_scan_peaks['temperature'], c_scan_peaks['peak_voltage'],
                        yerr=err_c,
                        fmt='s-', label='c_scan', color=COLOR_C_AXIS,
                        linewidth=2, markersize=8, capsize=3)

        ax.set_xlabel('$T$ (K)')
        ax.set_ylabel(r'$V_\mathrm{bias}^\mathrm{max(MR)}$ (V)')
        ax.grid(False)
        ax.legend()
        plt.tight_layout()
        plt.show()

        # Numerical results below the figure
        cols = ['temperature', 'scan_type', 'peak_voltage', 'peak_voltage_error',
                'method', 'chi2_red']
        print("Bias of the MR maximum with uncertainty "
              "(weighted parabola vertex, or plateau fallback):")
        print(df_peak_tmr.sort_values(['temperature', 'scan_type'])[cols]
              .to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
        unresolved = df_peak_tmr.loc[~df_peak_tmr['resolved'], 'temperature'].unique()
        print(f"\nparabola accepted for {int(df_peak_tmr['resolved'].sum())} of "
              f"{len(df_peak_tmr)} points; fallback used at T = "
              f"{sorted(unresolved)} K, where the MR maximum is not an interior "
              f"peak (below ~10 K the noise cut removes most of the sweep, above "
              f"~100 K the maximum sits on the zero-bias cusp).")
        print("\nPoint-to-point scatter over "
              f"{SMOOTH_T_RANGE_K[0]:.0f}-{SMOOTH_T_RANGE_K[1]:.0f} K, applied as "
              "the floor on the plotted error bars:")
        for name, s in peak_scatter.items():
            print(f"  {name}: {1000 * s:5.1f} mV  "
                  f"(median propagated error "
                  f"{1000 * df_peak_tmr.loc[df_peak_tmr['scan_type'] == name, 'peak_voltage_error'].median():5.1f} mV)")
    else:
        print("No peak TMR data found after filtering.")
else:
    print("df_combined not found in namespace.")


In [ ]:
import numpy as np
import pandas as pd

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

temperatures = [100, 90, 80, 70, 60, 50, 30, 20]

I_COL = 'I_apar (A)'
I_ERR_COL = 'I_apar_error (A)'

# Peak-MR biases with their uncertainties, as computed in section 6.
df_peaks = pd.concat([pd.DataFrame(peak_data_pos), pd.DataFrame(peak_data_neg)],
                     ignore_index=True)


def get_fn_coords(data, peak_voltage, peak_voltage_error):
    """FN coordinates (1/V, ln|I|/V^2) at the peak bias, with both error bars.

    The horizontal error comes from the peak-bias uncertainty via
    d(1/V) = sigma_V / V^2; the vertical error is the relative current error.
    """
    # Find the row closest to peak_voltage
    idx = (data['Voltage (V)'] - peak_voltage).abs().idxmin()
    V = data.loc[idx, 'Voltage (V)']
    I = data.loc[idx, I_COL]
    I_err = data.loc[idx, I_ERR_COL]
    if abs(V) >= 0.1 and I != 0:
        return (1 / V, np.log(abs(I) / V**2),
                peak_voltage_error / V**2, abs(I_err / I))
    return None, None, None, None


for scan_type, ax in [('b_scan', ax1), ('c_scan', ax2)]:

    # --- FN curves ---
    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type) &
            (df_combined['Voltage (V)'].abs() >= 0.02)
        ].copy()
        if len(data) > 0:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            x = 1 / data['Voltage (V)']
            y = np.log(data[I_COL].abs() / data['Voltage (V)']**2)
            yerr = fn_ordinate_error(data[I_COL], data[I_ERR_COL])
            # 1/V is discontinuous across V = 0, so caps are used rather than a
            # filled band, which would bridge the gap between the two branches.
            ax.errorbar(x, y, yerr=yerr, fmt='o', color=color, ecolor=color,
                        elinewidth=0.6, capsize=0, alpha=0.7, label=f'{temp}K')

    # --- Peak TMR markers ---
    for i, temp in enumerate(temperatures):
        data_full = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type)
        ].copy()
        if len(data_full) == 0:
            continue

        rows = df_peaks[(df_peaks['temperature'] == temp) &
                        (df_peaks['scan_type'] == scan_type)]
        for _, row in rows.iterrows():
            x_pk, y_pk, xerr, yerr = get_fn_coords(data_full, row['peak_voltage'],
                                                   row['peak_voltage_error'])
            if x_pk is not None:
                ax.errorbar(x_pk, y_pk, xerr=xerr, yerr=yerr,
                            fmt='*', markersize=14, markerfacecolor='white',
                            markeredgecolor='#D55E00', markeredgewidth=0.8,
                            ecolor='#D55E00', elinewidth=1.0, capsize=3, zorder=5)

    ax.set_xlabel(r'$1/V_\mathrm{bias}$ (V$^{-1}$)')
    ax.set_ylabel(r'$\ln(|I_\mathrm{AFM}|/V_\mathrm{bias}^2)$')
    ax.grid(False)
    ax.legend()

# Add a legend entry for the peak markers
from matplotlib.lines import Line2D
marker_legend = Line2D([0], [0], marker='*', color='w', markerfacecolor='white',
                        markeredgecolor=COLOR_C_AXIS, markersize=12, label='Peak MR voltage')
ax1.legend(handles=ax1.get_legend_handles_labels()[0] + [marker_legend],
           labels=ax1.get_legend_handles_labels()[1] + ['Peak MR voltage'])
ax2.legend(handles=ax2.get_legend_handles_labels()[0] + [marker_legend],
           labels=ax2.get_legend_handles_labels()[1] + ['Peak MR voltage'])

plt.tight_layout()
# Identify the figure in the cell output rather than with on-figure titles
print('Fowler-Nordheim plot (AFM) — left: b_scan, right: c_scan')
plt.show()


In [ ]:
import numpy as np
import pandas as pd

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

temperatures = [100, 90, 80, 70, 60, 50, 30, 20]

I_COL = 'I_par (A)'
I_ERR_COL = 'I_par_error (A)'

# Peak-MR biases with their uncertainties, as computed in section 6.
df_peaks = pd.concat([pd.DataFrame(peak_data_pos), pd.DataFrame(peak_data_neg)],
                     ignore_index=True)


def get_fn_coords(data, peak_voltage, peak_voltage_error):
    """FN coordinates (1/V, ln|I|/V^2) at the peak bias, with both error bars.

    The horizontal error comes from the peak-bias uncertainty via
    d(1/V) = sigma_V / V^2; the vertical error is the relative current error.
    """
    # Find the row closest to peak_voltage
    idx = (data['Voltage (V)'] - peak_voltage).abs().idxmin()
    V = data.loc[idx, 'Voltage (V)']
    I = data.loc[idx, I_COL]
    I_err = data.loc[idx, I_ERR_COL]
    if abs(V) >= 0.1 and I != 0:
        return (1 / V, np.log(abs(I) / V**2),
                peak_voltage_error / V**2, abs(I_err / I))
    return None, None, None, None


for scan_type, ax in [('b_scan', ax1), ('c_scan', ax2)]:

    # --- FN curves ---
    for i, temp in enumerate(temperatures):
        data = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type) &
            (df_combined['Voltage (V)'].abs() >= 0.02)
        ].copy()
        if len(data) > 0:
            color = OKABE_ITO_CYCLE[i % len(OKABE_ITO_CYCLE)]
            x = 1 / data['Voltage (V)']
            y = np.log(data[I_COL].abs() / data['Voltage (V)']**2)
            yerr = fn_ordinate_error(data[I_COL], data[I_ERR_COL])
            # 1/V is discontinuous across V = 0, so caps are used rather than a
            # filled band, which would bridge the gap between the two branches.
            ax.errorbar(x, y, yerr=yerr, fmt='o', color=color, ecolor=color,
                        elinewidth=0.6, capsize=0, alpha=0.7, label=f'{temp}K')

    # --- Peak TMR markers ---
    for i, temp in enumerate(temperatures):
        data_full = df_combined[
            (df_combined['temperature'] == temp) &
            (df_combined['scan_type'] == scan_type)
        ].copy()
        if len(data_full) == 0:
            continue

        rows = df_peaks[(df_peaks['temperature'] == temp) &
                        (df_peaks['scan_type'] == scan_type)]
        for _, row in rows.iterrows():
            x_pk, y_pk, xerr, yerr = get_fn_coords(data_full, row['peak_voltage'],
                                                   row['peak_voltage_error'])
            if x_pk is not None:
                ax.errorbar(x_pk, y_pk, xerr=xerr, yerr=yerr,
                            fmt='*', markersize=14, markerfacecolor='white',
                            markeredgecolor='#D55E00', markeredgewidth=0.8,
                            ecolor='#D55E00', elinewidth=1.0, capsize=3, zorder=5)

    ax.set_xlabel(r'$1/V_\mathrm{bias}$ (V$^{-1}$)')
    ax.set_ylabel(r'$\ln(|I_\mathrm{FM}|/V_\mathrm{bias}^2)$')
    ax.grid(False)
    ax.legend()

# Add a legend entry for the peak markers
from matplotlib.lines import Line2D
marker_legend = Line2D([0], [0], marker='*', color='w', markerfacecolor='white',
                        markeredgecolor=COLOR_C_AXIS, markersize=12, label='Peak MR voltage')
ax1.legend(handles=ax1.get_legend_handles_labels()[0] + [marker_legend],
           labels=ax1.get_legend_handles_labels()[1] + ['Peak MR voltage'])
ax2.legend(handles=ax2.get_legend_handles_labels()[0] + [marker_legend],
           labels=ax2.get_legend_handles_labels()[1] + ['Peak MR voltage'])

plt.tight_layout()
# Identify the figure in the cell output rather than with on-figure titles
print('Fowler-Nordheim plot (FM) — left: b_scan, right: c_scan')
plt.show()


In [ ]:
# Plot maximum TMR for positive bias voltage and maximum TMR for negative bias voltage
if peak_data_pos :
    fig, ax = plt.subplots(figsize=(6, 5), dpi=300)

    max_tmr_scatter = {}
    if peak_data_pos:
        df_pos = pd.DataFrame(peak_data_pos)
        b_scan_pos = df_pos[df_pos['scan_type'] == 'b_scan'].sort_values('temperature')
        c_scan_pos = df_pos[df_pos['scan_type'] == 'c_scan'].sort_values('temperature')

        # max_tmr_error is TMR_Error at the peak bias, scaled to the percent axis.
        # That error is a standard error of the mean over field points that are
        # correlated inside a magnetic state, so on its own it sits far below the
        # scatter of the series from one temperature to the next. It is therefore
        # raised to that measured scatter wherever it falls below it.
        if not b_scan_pos.empty:
            mr_b = 100 * b_scan_pos['max_tmr'] - 100
            err_b, max_tmr_scatter['b_scan'] = with_trend_floor(
                100 * b_scan_pos['max_tmr_error'], b_scan_pos['temperature'], mr_b,
                x_range=SMOOTH_T_RANGE_K)
            ax.errorbar(b_scan_pos['temperature'], mr_b, yerr=err_b,
                        fmt='o-', label='$b$ scan', color=COLOR_B_AXIS,
                        linewidth=2, markersize=8, capsize=3)
        if not c_scan_pos.empty:
            mr_c = 100 * c_scan_pos['max_tmr'] - 100
            err_c, max_tmr_scatter['c_scan'] = with_trend_floor(
                100 * c_scan_pos['max_tmr_error'], c_scan_pos['temperature'], mr_c,
                x_range=SMOOTH_T_RANGE_K)
            ax.errorbar(c_scan_pos['temperature'], mr_c, yerr=err_c,
                        fmt='s-', label='$c$ scan', color=COLOR_C_AXIS,
                        linewidth=2, markersize=8, capsize=3)


    ax.set_xlabel('$T$ (K)')
    ax.set_ylabel('max(MR ratio) (%)')

    ax.set_xlim(15, 165)
    ax.set_ylim(0, 400)
    #ax.grid(False)

    # Place legend outside to avoid obscuring data
    ax.legend( loc='upper left')

    plt.tight_layout()
    plt.show()

    # Error-bar magnitudes behind the figure above
    print("max(MR ratio): propagated error, and the error bar actually plotted")
    for name, frame in (('b_scan', b_scan_pos), ('c_scan', c_scan_pos)):
        if frame.empty:
            continue
        mr = 100 * frame['max_tmr'] - 100
        err, _ = with_trend_floor(100 * frame['max_tmr_error'],
                                  frame['temperature'], mr,
                                  x_range=SMOOTH_T_RANGE_K)
        for (_, row), e in zip(frame.sort_values('temperature').iterrows(), err):
            print(f"  {name} {int(row['temperature']):3d} K: "
                  f"{100*row['max_tmr']-100:7.1f} %,  propagated "
                  f"{100*row['max_tmr_error']:5.2f} %,  plotted +- {e:5.2f} %")
    print("\nPoint-to-point scatter over "
          f"{SMOOTH_T_RANGE_K[0]:.0f}-{SMOOTH_T_RANGE_K[1]:.0f} K, applied as the "
          "floor on the plotted error bars:")
    for name, s in max_tmr_scatter.items():
        print(f"  {name}: {s:5.2f} %")

else:
    print("Required data not found. Please run the previous cell first.")
